# $\newcommand{\ds}{\displaystyle}$ Logistic Regression Classifier with $\ell_1$ Regularization

- Here we use CVXPY to train a logistic regression classifier with $\ds\ell_1$ regularization
- We are given data $\ds(x_i,\,y_i),\;i=1,\ldots, m$
- The $x_i \in \mathbb{R}^n$ are feature vectors, while the $y_i \in \{0,\,1\}$ are associated boolean outcome
- Goal: To construct a linear classifier
  \begin{align*}
    \hat y = \mathbb{1}\big\{\beta^\top x > 0\big\} = \begin{cases}1 & \text{if }\;\beta^\top x > 0 \\ 0 & \text{otherwise}\end{cases}
  \end{align*}
- Model the posterior probabilities of the classes given the data with
\begin{align*}
  \log \frac{\mathsf{P} (Y = 1 \mid X = x)}{\mathsf{P} (Y = 0 \mid X = x)} = \beta^\top x.
\end{align*}
which means
\begin{align*}
  \mathsf{P} (Y = 1 \mid X = x) &= \frac{\exp(\beta^\top x)}{1 + \exp(\beta^\top x)} \\
  \mathsf{P} (Y = 0 \mid X = x) &= \frac{1}{1 + \exp(\beta^\top x)}
\end{align*}
- Fit $\beta$ by maximizing the log-likelihood of the data plus a regularization term $\lambda \|\beta\|_1$ with $\lambda > 0$:
\begin{align*}
  \ell(\beta) = \sum_{i=1}^m\big(y_i\,\beta^\top x_i - \log(1 + \exp (\beta^\top x_i))\big) - \lambda \|\beta\|_1.
\end{align*}
- This is a convex optimization problem for $\ell$ is a concave function of $\beta$

In [ ]:
import cvxpy as cp
import numpy as np
import matplotlib.pyplot as plt

## Example

- Generate data with $n=50$ features by randomly choosing $x_i$ and supplying a sparse $\beta_{\mathrm{true}} \in \mathbb{R}^n$
- Set $\ds y_i = \mathbb{1}\big\{\beta_{\mathrm{true}}^\top x_i + z_i > 0\big\}$ where $z_i$ are i.i.d. normal random variables
- Divide the data into training and test sets with $m=50$ examples each

In [ ]:
np.random.seed(1)
n = 50
m = 50
def sigmoid(z):
  return 1 / (1 + np.exp(-z))

beta_true = np.array([1, 0.5, -0.5] + [0] * (n - 3))
X = (np.random.random((m, n)) - 0.5) * 10
Y = np.round(sigmoid(X @ beta_true + np.random.randn(m) * 0.5))

X_test = (np.random.random((2 * m, n)) - 0.5) * 10
Y_test = np.round(sigmoid(X_test @ beta_true + np.random.randn(2 * m) * 0.5))

- Solve the optimization problem for a range of $\lambda$ to compute a trade-off curve
- Plot the train and test error over the trade-off curve
- A reasonable choice of $\lambda$ is the value that minimizes the test error

In [ ]:
beta = cp.Variable(n)
lambd = cp.Parameter(nonneg=True)
log_likelihood = cp.sum(cp.multiply(Y, X @ beta) - cp.logistic(X @ beta))
problem = cp.Problem(cp.Maximize(log_likelihood / m - lambd * cp.norm(beta, 1)))

In [ ]:
def error(scores, labels):
    scores[scores > 0] = 1
    scores[scores <= 0] = 0
    return np.sum(np.abs(scores - labels)) / float(np.size(labels))

In [ ]:
trials = 100
train_error = np.zeros(trials)
test_error = np.zeros(trials)
lambda_vals = np.logspace(-2, 0, trials)
beta_vals = []
for i in range(trials):
    lambd.value = lambda_vals[i]
    problem.solve()
    train_error[i] = error((X @ beta).value, Y)
    test_error[i] = error((X_test @ beta).value, Y_test)
    beta_vals.append(beta.value)

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = "svg"

plt.plot(lambda_vals, train_error, label="Train error")
plt.plot(lambda_vals, test_error, label="Test error")
plt.xscale("log")
plt.legend(loc="upper left")
plt.xlabel(r"$\lambda$", fontsize=16)
plt.show()

- Plot the regularization path --- the $\beta_i$ versus $\lambda$
- A few features remain non-zero longer for larger $\lambda$; these features are the most important 

In [ ]:
for i in range(n):
    plt.plot(lambda_vals, [wi for wi in beta_vals])
plt.xlabel(r"$\lambda$", fontsize=16)
plt.xscale("log")

- Plot the true $\beta$ versus reconstructed $\beta$ as chosen to minimize error on the test set
- The non-zero coefficients are reconstructed with good accuracy
- A few values in the reconstructed $\beta$ that are non-zero but should be zero

In [ ]:
idx = np.argmin(test_error)
plt.plot(beta_true, label=r"True $\beta$")
plt.plot(beta_vals[idx], label=r"Reconstructed $\beta$")
plt.xlabel(r"$i$", fontsize=16)
plt.ylabel(r"$\beta_i$", fontsize=16)
plt.legend(loc="upper right")